# Spark Session

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from datetime import datetime, timedelta
import os

# Database configuration
DB_CONFIG = {
    'host': '10.205.161.118',
    'port': '5432',
    'database': 'db_fraud',
    'user': 'dfstechbi',
    'password': 'DfsTeChB1@923'
}

# JDBC Configuration
jdbc_driver_path = "/root/research-dir/dev/jazzcash-fraud-detection/utils/postgresql-42.7.1.jar"
jdbc_url = f"jdbc:postgresql://{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"

# Optimized JDBC properties
properties = {
    "user": DB_CONFIG['user'],
    "password": DB_CONFIG['password'],
    "driver": "org.postgresql.Driver",
    "fetchsize": "10000",
    "batchsize": "15000",
    "isolationLevel": "READ_UNCOMMITTED",
    "queryTimeout": "1200",
    "loginTimeout": "60",
    "socketTimeout": "1200",
    "tcpKeepAlive": "true",
    "prepareThreshold": "5",
    "reWriteBatchedInserts": "true",
    "defaultRowFetchSize": "10000"
}

print("🚀 Creating optimized Spark session for asymmetric cluster...")
print("📊 Cluster Resources:")
print("   • Worker 1: 32 cores, 120GB RAM")
print("   • Worker 2: 32 cores, 110GB RAM")
print("   • Total: 64 cores, 230GB RAM")

# Create Spark session with optimized configuration for asymmetric cluster
spark = SparkSession.builder \
    .appName("Feature-Job-Optimized") \
    .master("spark://dfs-ai-app2:7077") \
    .config("spark.jars", jdbc_driver_path) \
    .config("spark.executor.instances", "12") \
    .config("spark.executor.cores", "5") \
    .config("spark.executor.memory", "17g") \
    .config("spark.executor.memoryOverhead", "3g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.memoryOverhead", "1g") \
    .config("spark.driver.cores", "2") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128MB") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.dynamicAllocation.enabled", "false") \
    .config("spark.shuffle.service.enabled", "false") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "120") \
    .config("spark.sql.autoBroadcastJoinThreshold", "100MB") \
    .config("spark.executor.extraJavaOptions", "-Xss4m") \
    .config("spark.driver.extraJavaOptions", "-Xss4m") \
    .getOrCreate()



# Set log level to reduce noise
spark.sparkContext.setLogLevel("WARN")

print("✅ Spark session created successfully!")
print(f"📱 Application ID: {spark.sparkContext.applicationId}")
print(f"🎯 Master: {spark.sparkContext.master}")
print(f"🔧 Configuration Summary:")
print(f"   • Executors: 12 instances")
print(f"   • Cores per executor: 5")
print(f"   • Memory per executor: 17GB + 3GB overhead")
print(f"   • Driver: 4GB + 1GB overhead, 2 cores")
print(f"   • Total executor cores: 60 (leaving 4 for OS)")
print(f"   • Total executor memory: ~204GB (leaving ~26GB for OS/overhead)")

🚀 Creating optimized Spark session for asymmetric cluster...
📊 Cluster Resources:
   • Worker 1: 32 cores, 120GB RAM
   • Worker 2: 32 cores, 110GB RAM
   • Total: 64 cores, 230GB RAM


25/10/23 10:53:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


✅ Spark session created successfully!
📱 Application ID: app-20251023105312-0036
🎯 Master: spark://dfs-ai-app2:7077
🔧 Configuration Summary:
   • Executors: 12 instances
   • Cores per executor: 5
   • Memory per executor: 17GB + 3GB overhead
   • Driver: 4GB + 1GB overhead, 2 cores
   • Total executor cores: 60 (leaving 4 for OS)
   • Total executor memory: ~204GB (leaving ~26GB for OS/overhead)


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 48324)
Traceback (most recent call last):
  File "/root/miniconda3/envs/fraud/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/root/miniconda3/envs/fraud/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/root/miniconda3/envs/fraud/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/root/miniconda3/envs/fraud/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/root/miniconda3/envs/fraud/lib/python3.10/site-packages/pyspark/accumulators.py", line 297, in handle
    poll(authenticate_and_accum_updates)
  File "/root/miniconda3/envs/fraud/lib/python3.10/site-packages/pyspark/accumulators.py", line 271, in poll
    if self.rfile in

# Load IAR

In [2]:
table_name = "public.stixor_iar"

print("🔧 Setting up date-based predicates...")

# Create predicates for optimized partitioning
predicates = []
start_date = datetime.strptime("2025-06-01", "%Y-%m-%d")
end_date = datetime.strptime("2025-07-05", "%Y-%m-%d")

current_date = start_date
while current_date <= end_date:
    date_str = current_date.strftime("%Y-%m-%d")
    predicates.append(f"data_date = '{date_str}'")
    current_date += timedelta(days=1)

# print(f"📊 Created {len(predicates)} predicates for date range")

print("\n🔗 Loading data from PostgreSQL...")

# Load data using predicate-based partitioning
df = spark.read.jdbc(
    url=jdbc_url,
    table=table_name,
    properties=properties,
    predicates=predicates
)

🔧 Setting up date-based predicates...

🔗 Loading data from PostgreSQL...


# Load Customer Senders

In [3]:
# Load July customer senders from saved parquet
df_july_customer_senders = spark.read.parquet("../data/july_2025_customer_senders")
print(f"📊 Loaded July customer senders: {df_july_customer_senders.count():,}")

📊 Loaded July customer senders: 22,579,490


In [13]:
# Take a random sample of 1,000,000 customers
df_july_customer_senders_sample = df_july_customer_senders.sample(fraction=0.1, seed=42).limit(1000000)
df_july_customer_senders_sample.cache()
sample_count = df_july_customer_senders_sample.count()
print(f"📊 Random sample of July customer senders: {sample_count:,}")

📊 Random sample of July customer senders: 1,000,000


25/10/23 11:28:36 WARN CacheManager: Asked to cache already cached data.


# FraudFeatureEngineer

In [14]:
# Replace the existing FraudFeatureEngineer class with this user-level version

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import datetime, timedelta
import numpy as np

# Define time windows in seconds
TIME_WINDOWS = {
    '1d': 86400,
    '5d': 432000,
    '10d': 864000,
    '20d': 1728000,
    '30d': 2592000
}

class UserFraudFeatureEngineer:
    """
    User-level feature engineering for fraud detection - creates one row per customer
    """
    
    def __init__(self, df):
        """
        Initialize with transaction dataframe
        Will aggregate all transactions per customer into user-level features
        """
        self.df = df
        
        # Convert timestamp to unix timestamp for window calculations
        self.df = self.df.withColumn('unix_timestamp', 
                                     F.unix_timestamp(F.col('trans_initiate_time')))
        
        # Create a clean amount column (handle nulls)
        self.df = self.df.withColumn('amount_clean',
                                     F.coalesce(F.col('trx_amt'), F.lit(0)))
        
        # Repartition by customer for better aggregation performance
        self.df = self.df.repartition(200, 'ac_from')
        
        # Store the current date for relative calculations (use max date in dataset)
        self.reference_date = self.df.agg(F.max('trans_initiate_time')).collect()[0][0]
        print(f"Using reference date for calculations: {self.reference_date}")
        
    def create_user_basic_features(self):
        """
        Create basic user-level aggregated features
        """
        print("Creating basic user-level features...")
        
        # Basic aggregations per user
        user_basic = self.df.groupBy('ac_from').agg(
            # Transaction counts and timing
            F.count('*').alias('total_transactions'),
            F.min('trans_initiate_time').alias('first_transaction_date'),
            F.max('trans_initiate_time').alias('last_transaction_date'),
            F.datediff(F.max('trans_initiate_time'), F.min('trans_initiate_time')).alias('account_age_days'),
            
            # Amount statistics
            F.sum('amount_clean').alias('total_amount'),
            F.avg('amount_clean').alias('avg_amount'),
            F.stddev('amount_clean').alias('std_amount'),
            F.min('amount_clean').alias('min_amount'),
            F.max('amount_clean').alias('max_amount'),
            F.expr('percentile_approx(amount_clean, 0.5)').alias('median_amount'),
            F.expr('percentile_approx(amount_clean, 0.25)').alias('q25_amount'),
            F.expr('percentile_approx(amount_clean, 0.75)').alias('q75_amount'),
            
            # Balance statistics
            F.avg('start_balance').alias('avg_start_balance'),
            F.avg('end_balance').alias('avg_end_balance'),
            F.min('start_balance').alias('min_start_balance'),
            F.max('end_balance').alias('max_end_balance'),
            F.avg(F.col('end_balance') - F.col('start_balance')).alias('avg_balance_change'),
            
            # Fee statistics
            F.sum(F.coalesce(F.col('fee'), F.lit(0))).alias('total_fees'),
            F.avg(F.coalesce(F.col('fee'), F.lit(0))).alias('avg_fee'),
            F.sum(F.coalesce(F.col('fed'), F.lit(0))).alias('total_fed'),
            F.avg(F.coalesce(F.col('fed'), F.lit(0))).alias('avg_fed'),
            
            # Channel and type diversity
            F.countDistinct('trx_channel').alias('unique_channels'),
            F.countDistinct('trx_type').alias('unique_transaction_types'),
            F.countDistinct('ac_to').alias('unique_recipients'),
            F.countDistinct(F.when(F.col('merchant_id').isNotNull(), F.col('merchant_id'))).alias('unique_merchants'),
            F.countDistinct(F.when(F.col('utility_company').isNotNull(), F.col('utility_company'))).alias('unique_utilities'),
        )
        
        # Add derived features
        user_basic = user_basic \
            .withColumn('transactions_per_day', 
                       F.when(F.col('account_age_days') > 0, 
                             F.col('total_transactions') / F.col('account_age_days'))
                       .otherwise(F.col('total_transactions'))) \
            .withColumn('amount_per_day',
                       F.when(F.col('account_age_days') > 0,
                             F.col('total_amount') / F.col('account_age_days'))
                       .otherwise(F.col('total_amount'))) \
            .withColumn('fee_to_amount_ratio',
                       F.when(F.col('total_amount') > 0,
                             F.col('total_fees') / F.col('total_amount'))
                       .otherwise(0)) \
            .withColumn('channel_diversity_ratio',
                       F.col('unique_channels') / F.col('total_transactions')) \
            .withColumn('recipient_diversity_ratio',
                       F.col('unique_recipients') / F.col('total_transactions')) \
            .withColumn('amount_coefficient_variation',
                       F.when(F.col('avg_amount') > 0,
                             F.col('std_amount') / F.col('avg_amount'))
                       .otherwise(0))
        
        self.user_features = user_basic
        return self
    
    def create_user_time_pattern_features(self):
        """
        Create time-based behavioral pattern features per user
        """
        print("Creating user time pattern features...")
        
        # Time pattern aggregations
        time_patterns = self.df.groupBy('ac_from').agg(
            # Hour patterns
            F.avg(F.hour('trans_initiate_time')).alias('avg_transaction_hour'),
            F.stddev(F.hour('trans_initiate_time')).alias('std_transaction_hour'),
            F.sum(F.when((F.hour('trans_initiate_time') >= 22) | 
                        (F.hour('trans_initiate_time') <= 6), 1).otherwise(0)).alias('night_transactions'),
            F.sum(F.when(F.hour('trans_initiate_time').between(9, 17), 1).otherwise(0)).alias('business_hour_transactions'),
            
            # Day patterns
            F.sum(F.when(F.dayofweek('trans_initiate_time').isin([1, 7]), 1).otherwise(0)).alias('weekend_transactions'),
            F.countDistinct(F.dayofweek('trans_initiate_time')).alias('unique_days_of_week'),
            F.countDistinct(F.hour('trans_initiate_time')).alias('unique_hours'),
            
            # Monthly patterns
            F.countDistinct(F.month('trans_initiate_time')).alias('unique_months'),
            F.countDistinct(F.dayofmonth('trans_initiate_time')).alias('unique_days_of_month'),
        )
        
        # Add derived time features
        time_patterns = time_patterns \
            .withColumn('night_transaction_ratio',
                       F.col('night_transactions') / F.col('total_transactions')) \
            .withColumn('business_hour_ratio',
                       F.col('business_hour_transactions') / F.col('total_transactions')) \
            .withColumn('weekend_transaction_ratio',
                       F.col('weekend_transactions') / F.col('total_transactions')) \
            .withColumn('time_diversity_score',
                       (F.col('unique_hours') / 24.0 + F.col('unique_days_of_week') / 7.0) / 2.0)
        
        # Join with main features
        self.user_features = self.user_features.join(time_patterns, on='ac_from', how='left')
        return self
    
    def create_user_channel_features(self):
        """
        Create channel usage pattern features per user
        """
        print("Creating user channel pattern features...")
        
        # Get top channels
        top_channels = self.df.select('trx_channel').filter(F.col('trx_channel').isNotNull()) \
                             .groupBy('trx_channel').count().orderBy(F.desc('count')).limit(10).collect()
        top_channels_list = [row['trx_channel'] for row in top_channels]
        
        # Channel pattern aggregations
        channel_aggs = [F.count('*').alias('total_transactions')]
        
        # Add aggregations for each top channel
        for channel in top_channels_list:
            safe_channel = channel.replace(' ', '_').replace('-', '_')[:15]
            channel_aggs.extend([
                F.sum(F.when(F.col('trx_channel') == channel, 1).otherwise(0)).alias(f'{safe_channel}_count'),
                F.sum(F.when(F.col('trx_channel') == channel, F.col('amount_clean')).otherwise(0)).alias(f'{safe_channel}_total_amount'),
                F.avg(F.when(F.col('trx_channel') == channel, F.col('amount_clean'))).alias(f'{safe_channel}_avg_amount')
            ])
        
        channel_patterns = self.df.groupBy('ac_from').agg(*channel_aggs)
        
        # Add channel ratios
        for channel in top_channels_list:
            safe_channel = channel.replace(' ', '_').replace('-', '_')[:15]
            channel_patterns = channel_patterns \
                .withColumn(f'{safe_channel}_usage_ratio',
                           F.col(f'{safe_channel}_count') / F.col('total_transactions_new'))
        
        # Join with main features (drop duplicate total_transactions column)
        # channel_patterns = channel_patterns.drop('total_transactions')
        self.user_features = self.user_features.join(channel_patterns, on='ac_from', how='left')
        return self
    
    def create_user_transaction_type_features(self):
        """
        Create transaction type pattern features per user
        """
        print("Creating user transaction type pattern features...")
        
        # Get top transaction types
        top_types = self.df.select('trx_type').filter(F.col('trx_type').isNotNull()) \
                          .groupBy('trx_type').count().orderBy(F.desc('count')).limit(10).collect()
        top_types_list = [row['trx_type'] for row in top_types]
        
        # Transaction type aggregations
        type_aggs = []
        for txn_type in top_types_list:
            safe_type = txn_type.replace(' ', '_').replace('-', '_').replace('/', '_')[:15]
            type_aggs.extend([
                F.sum(F.when(F.col('trx_type') == txn_type, 1).otherwise(0)).alias(f'type_{safe_type}_count'),
                F.sum(F.when(F.col('trx_type') == txn_type, F.col('amount_clean')).otherwise(0)).alias(f'type_{safe_type}_total_amount'),
                F.avg(F.when(F.col('trx_type') == txn_type, F.col('amount_clean'))).alias(f'type_{safe_type}_avg_amount')
            ])
        
        type_patterns = self.df.groupBy('ac_from').agg(*type_aggs)
        
        # Add type ratios
        for txn_type in top_types_list:
            safe_type = txn_type.replace(' ', '_').replace('-', '_').replace('/', '_')[:15]
            type_patterns = type_patterns \
                .withColumn(f'type_{safe_type}_usage_ratio',
                           F.col(f'type_{safe_type}_count') / F.col('total_transactions'))
        
        self.user_features = self.user_features.join(type_patterns, on='ac_from', how='left')
        return self
    
    def create_user_status_features(self):
        """
        Create transaction status pattern features per user
        """
        print("Creating user transaction status pattern features...")
        
        status_patterns = self.df.groupBy('ac_from').agg(
            # Success/failure patterns
            F.sum(F.when(F.upper(F.col('trx_status')).isin(['COMPLETED', 'SUCCESS', 'SUCCESSFUL']), 1)
                 .otherwise(0)).alias('successful_transactions'),
            F.sum(F.when(F.upper(F.col('trx_status')).isin(['CANCELLED', 'DECLINED', 'FAILED']), 1)
                 .otherwise(0)).alias('failed_transactions'),
            
            # Amount patterns by status
            F.sum(F.when(F.upper(F.col('trx_status')).isin(['COMPLETED', 'SUCCESS', 'SUCCESSFUL']), 
                        F.col('amount_clean')).otherwise(0)).alias('successful_amount'),
            F.sum(F.when(F.upper(F.col('trx_status')).isin(['CANCELLED', 'DECLINED', 'FAILED']), 
                        F.col('amount_clean')).otherwise(0)).alias('failed_amount'),
            
            # Status diversity
            F.countDistinct('trx_status').alias('unique_statuses')
        )
        
        # Add derived status features
        status_patterns = status_patterns \
            .withColumn('success_rate',
                       F.col('successful_transactions') / F.col('total_transactions')) \
            .withColumn('failure_rate',
                       F.col('failed_transactions') / F.col('total_transactions')) \
            .withColumn('avg_successful_amount',
                       F.when(F.col('successful_transactions') > 0,
                             F.col('successful_amount') / F.col('successful_transactions'))
                       .otherwise(0)) \
            .withColumn('avg_failed_amount',
                       F.when(F.col('failed_transactions') > 0,
                             F.col('failed_amount') / F.col('failed_transactions'))
                       .otherwise(0))
        
        self.user_features = self.user_features.join(status_patterns, on='ac_from', how='left')
        return self
    
    def create_user_merchant_utility_features(self):
        """
        Create merchant and utility usage pattern features per user
        """
        print("Creating user merchant and utility pattern features...")
        
        merchant_utility_patterns = self.df.groupBy('ac_from').agg(
            # Merchant patterns
            F.sum(F.when(F.col('merchant_id').isNotNull(), 1).otherwise(0)).alias('merchant_transactions'),
            F.sum(F.when(F.col('merchant_id').isNotNull(), F.col('amount_clean')).otherwise(0)).alias('merchant_total_amount'),
            F.avg(F.when(F.col('merchant_id').isNotNull(), F.col('amount_clean'))).alias('merchant_avg_amount'),
            
            # Utility patterns
            F.sum(F.when(F.col('utility_company').isNotNull(), 1).otherwise(0)).alias('utility_transactions'),
            F.sum(F.when(F.col('utility_company').isNotNull(), F.col('amount_clean')).otherwise(0)).alias('utility_total_amount'),
            F.avg(F.when(F.col('utility_company').isNotNull(), F.col('amount_clean'))).alias('utility_avg_amount'),
        )
        
        # Add derived features
        merchant_utility_patterns = merchant_utility_patterns \
            .withColumn('merchant_transaction_ratio',
                       F.col('merchant_transactions') / F.col('total_transactions')) \
            .withColumn('utility_transaction_ratio',
                       F.col('utility_transactions') / F.col('total_transactions')) \
            .withColumn('merchant_diversity_ratio',
                       F.col('unique_merchants') / F.greatest(F.col('merchant_transactions'), F.lit(1))) \
            .withColumn('utility_diversity_ratio',
                       F.col('unique_utilities') / F.greatest(F.col('utility_transactions'), F.lit(1)))
        
        self.user_features = self.user_features.join(merchant_utility_patterns, on='ac_from', how='left')
        return self
    
    def create_user_velocity_features(self):
        """
        Create velocity and temporal pattern features per user
        """
        print("Creating user velocity pattern features...")
        
        # Calculate transaction intervals
        df_with_intervals = self.df \
            .withColumn('prev_transaction_time',
                       F.lag('unix_timestamp').over(Window.partitionBy('ac_from').orderBy('unix_timestamp'))) \
            .withColumn('transaction_interval_hours',
                       (F.col('unix_timestamp') - F.col('prev_transaction_time')) / 3600)
        
        velocity_patterns = df_with_intervals.groupBy('ac_from').agg(
            # Transaction timing patterns
            F.avg('transaction_interval_hours').alias('avg_transaction_interval_hours'),
            F.stddev('transaction_interval_hours').alias('std_transaction_interval_hours'),
            F.min('transaction_interval_hours').alias('min_transaction_interval_hours'),
            F.max('transaction_interval_hours').alias('max_transaction_interval_hours'),
            
            # Burst patterns (transactions within short time windows)
            F.sum(F.when(F.col('transaction_interval_hours') < 1, 1).otherwise(0)).alias('burst_transactions_1h'),
            F.sum(F.when(F.col('transaction_interval_hours') < 24, 1).otherwise(0)).alias('burst_transactions_24h'),
            
            # Recent activity patterns (last 30 days from reference date)
            F.sum(F.when(F.datediff(F.lit(self.reference_date), F.col('trans_initiate_time')) <= 30, 1)
                 .otherwise(0)).alias('transactions_last_30d'),
            F.sum(F.when(F.datediff(F.lit(self.reference_date), F.col('trans_initiate_time')) <= 7, 1)
                 .otherwise(0)).alias('transactions_last_7d'),
            F.sum(F.when(F.datediff(F.lit(self.reference_date), F.col('trans_initiate_time')) <= 1, 1)
                 .otherwise(0)).alias('transactions_last_1d'),
        )
        
        # Add derived velocity features
        velocity_patterns = velocity_patterns \
            .withColumn('burst_ratio_1h',
                       F.col('burst_transactions_1h') / F.col('total_transactions')) \
            .withColumn('burst_ratio_24h',
                       F.col('burst_transactions_24h') / F.col('total_transactions')) \
            .withColumn('recent_activity_ratio_30d',
                       F.col('transactions_last_30d') / F.col('total_transactions')) \
            .withColumn('recent_activity_ratio_7d',
                       F.col('transactions_last_7d') / F.col('total_transactions')) \
            .withColumn('transaction_regularity_score',
                       F.when(F.col('std_transaction_interval_hours') > 0,
                             1.0 / (1.0 + F.col('std_transaction_interval_hours') / 
                                   F.greatest(F.col('avg_transaction_interval_hours'), F.lit(1))))
                       .otherwise(1.0))
        
        self.user_features = self.user_features.join(velocity_patterns, on='ac_from', how='left')
        return self
    
    def create_user_risk_scores(self):
        """
        Create composite risk scores per user
        """
        print("Creating user risk scores...")
        
        self.user_features = self.user_features \
            .withColumn('velocity_risk_score',
                       F.when(F.col('transactions_per_day') > 10, 3)
                       .when(F.col('transactions_per_day') > 5, 2)
                       .when(F.col('transactions_per_day') > 2, 1)
                       .otherwise(0)) \
            .withColumn('amount_risk_score',
                       F.when(F.col('amount_coefficient_variation') > 2, 3)
                       .when(F.col('amount_coefficient_variation') > 1.5, 2)
                       .when(F.col('amount_coefficient_variation') > 1, 1)
                       .otherwise(0)) \
            .withColumn('behavioral_risk_score',
                       (F.when(F.col('night_transaction_ratio') > 0.3, 1).otherwise(0) +
                        F.when(F.col('failure_rate') > 0.1, 1).otherwise(0) +
                        F.when(F.col('burst_ratio_1h') > 0.1, 1).otherwise(0) +
                        F.when(F.col('channel_diversity_ratio') > 0.5, 1).otherwise(0))) \
            .withColumn('overall_user_risk_score',
                       (F.col('velocity_risk_score') * 0.3 +
                        F.col('amount_risk_score') * 0.3 +
                        F.col('behavioral_risk_score') * 0.4))
        
        return self
    
    def execute_user_pipeline(self):
        """
        Execute the complete user-level feature engineering pipeline
        """
        print("\n" + "="*80)
        print("Starting User-Level Fraud Detection Feature Engineering Pipeline")
        print("="*80 + "\n")
        
        start_time = datetime.now()
        
        # Execute all user-level feature creation methods
        self.create_user_basic_features()
        # feature_engineer.df = feature_engineer.df.localCheckpoint()
        
        # self.create_user_time_pattern_features()
        # self.create_user_channel_features()
        # self.create_user_transaction_type_features()
        # self.create_user_status_features()
        # self.create_user_merchant_utility_features()
        # self.create_user_velocity_features()
        # self.create_user_risk_scores()
        
        # print("     - Final checkpoint before save...")
        # df_batch_features = feature_engineer.df.localCheckpoint()
        # Cache the result
        self.user_features.cache()
        user_count = self.user_features.count()  # Trigger cache
        
        end_time = datetime.now()
        
        print("\n" + "="*80)
        print(f"User-Level Feature Engineering Pipeline Completed Successfully!")
        print(f"Execution Time: {end_time - start_time}")
        print(f"Total Users Processed: {user_count:,}")
        print(f"Total Features Created: {len(self.user_features.columns)}")
        print("="*80 + "\n")
        
        return self.user_features

# Update the batch processing function to use the new user-level feature engineer
def process_customer_batches_user_level(df_customers, df_transactions, batch_size=10000, target_table="customer_fraud_features_user_level"):
    """
    Optimized batch processing for user-level customer features
    """
    
    total_customers = df_customers.count()
    total_batches = math.ceil(total_customers / batch_size)
    
    print(f"🚀 Starting user-level batch processing:")
    print(f"   • Total customers: {total_customers:,}")
    print(f"   • Batch size: {batch_size:,}")
    print(f"   • Total batches: {total_batches}")
    print(f"   • Target table: {target_table}")
    print("="*80)
    
    # Create target table schema for user-level features
    create_user_target_table(target_table)
    
    # Pre-process customers with hash-based partitioning
    df_customers_hashed = df_customers.withColumn(
        "hash_mod", 
        F.abs(F.hash(F.col("ac_from"))) % total_batches
    )
    
    df_customers_hashed.cache()
    df_customers_hashed.count()
    
    successful_batches = 0
    total_users_processed = 0
    
    for batch_num in range(total_batches):
        batch_start_time = datetime.now()
        
        print(f"\n📦 Processing Batch {batch_num + 1}/{total_batches}")
        
        # Get customer batch
        customer_batch = df_customers_hashed.filter(F.col("hash_mod") == batch_num).select("ac_from")
        batch_customer_count = customer_batch.count()
        
        if batch_customer_count == 0:
            continue
        
        print(f"   • Customers in batch: {batch_customer_count:,}")
        
        # Filter transactions for this batch
        df_batch_transactions = df_transactions.join(customer_batch, on="ac_from", how="inner")
        batch_txn_count = df_batch_transactions.count()
        
        print(f"   • Transactions in batch: {batch_txn_count:,}")
        
        if batch_txn_count == 0:
            continue
        
        try:
            # Create user-level features
            print("   • Creating user-level features...")
            user_feature_engineer = UserFraudFeatureEngineer(df_batch_transactions)
            df_user_features = user_feature_engineer.execute_user_pipeline()
            
            # Add batch metadata
            df_user_features = df_user_features \
                .withColumn("batch_number", F.lit(batch_num + 1)) \
                .withColumn("processing_timestamp", F.current_timestamp()) \
                .withColumn("created_at", F.current_timestamp())
            
            feature_count = len(df_user_features.columns)
            user_count = df_user_features.count()
            
            # Write to database
            print(f"   • Writing {user_count:,} users to database...")
            
            write_properties = {
                "user": DB_CONFIG['user'],
                "password": DB_CONFIG['password'],
                "driver": "org.postgresql.Driver",
                "batchsize": "5000",
                "isolationLevel": "READ_UNCOMMITTED",
                "reWriteBatchedInserts": "true",
                "stringtype": "unspecified"
            }
            
            df_user_features.write \
                .jdbc(
                    url=jdbc_url,
                    table=f"public.{target_table}",
                    mode="append",
                    properties=write_properties
                )
            
            total_users_processed += user_count
            successful_batches += 1
            
            batch_end_time = datetime.now()
            batch_duration = batch_end_time - batch_start_time
            
            print(f"   ✅ Batch {batch_num + 1} completed!")
            print(f"   • Duration: {batch_duration}")
            print(f"   • Users processed: {user_count:,}")
            print(f"   • Features created: {feature_count}")
            
            # Clean up
            customer_batch.unpersist()
            df_batch_transactions.unpersist()
            df_user_features.unpersist()
            
        except Exception as e:
            print(f"   ❌ Error processing batch {batch_num + 1}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    df_customers_hashed.unpersist()
    
    print("\n" + "="*80)
    print("🎉 User-level batch processing completed!")
    print(f"   • Successful batches: {successful_batches}/{total_batches}")
    print(f"   • Total users processed: {total_users_processed:,}")
    print(f"   • Target table: public.{target_table}")
    print("="*80)

def create_user_target_table(table_name):
    """
    Create target table for user-level features
    """
    print(f"🔧 Setting up user-level target table: {table_name}")
    # Table will be created automatically on first write with proper schema inference
    
# Updated function call for user-level processing
print("Starting user-level batch feature engineering...")
process_customer_batches_user_level(
    df_customers=df_july_customer_senders_sample,
    df_transactions=df,
    batch_size=10000,  # Smaller batch size since we're aggregating
    target_table="customer_fraud_basic_features"
)

Starting user-level batch feature engineering...
🚀 Starting user-level batch processing:
   • Total customers: 1,000,000
   • Batch size: 10,000
   • Total batches: 100
   • Target table: customer_fraud_basic_features
🔧 Setting up user-level target table: customer_fraud_basic_features

📦 Processing Batch 1/100
   • Customers in batch: 22,323


25/10/23 11:28:57 WARN CacheManager: Asked to cache already cached data.


   • Transactions in batch: 212,114
   • Creating user-level features...


Using reference date for calculations: 2025-07-05 23:59:36

Starting User-Level Fraud Detection Feature Engineering Pipeline

Creating basic user-level features...



User-Level Feature Engineering Pipeline Completed Successfully!
Execution Time: 0:01:02.385601
Total Users Processed: 18,086
Total Features Created: 33

   • Writing 18,086 users to database...
   ✅ Batch 1 completed!
   • Duration: 0:01:50.376009
   • Users processed: 18,086
   • Features created: 36

📦 Processing Batch 2/100
   • Customers in batch: 11,247


...

   ✅ Batch 36 completed!
   • Duration: 0:05:20.086679
   • Users processed: 9,184
   • Features created: 36

📦 Processing Batch 37/100
   • Customers in batch: 11,214


   • Transactions in batch: 101,848
   • Creating user-level features...


Using reference date for calculations: 2025-07-05 23:58:50

Starting User-Level Fraud Detection Feature Engineering Pipeline

Creating basic user-level features...


ERROR:root:KeyboardInterrupt while sending command.               (0 + 30) / 35]
Traceback (most recent call last):
  File "/root/miniconda3/envs/fraud/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/root/miniconda3/envs/fraud/lib/python3.10/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/root/miniconda3/envs/fraud/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt
ERROR:py4j.clientserver:Exception occurred while shutting down connection
Traceback (most recent call last):
  File "/root/miniconda3/envs/fraud/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/root/miniconda3/envs/fraud/lib/python3.10/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(se

KeyboardInterrupt: 

# Batch Process Features

In [ ]:
import math
from datetime import datetime

def process_customer_batches_optimized(df_customers, df_transactions, batch_size=1000000, output_path="../data/customer_features_batched"):
    """
    Optimized batch processing for customer features
    """
    
    total_customers = df_customers.count()
    total_batches = math.ceil(total_customers / batch_size)
    
    print(f"🚀 Starting optimized batch processing:")
    print(f"   • Total customers: {total_customers:,}")
    print(f"   • Batch size: {batch_size:,}")
    print(f"   • Total batches: {total_batches}")
    print(f"   • Output path: {output_path}")
    print("="*80)
    
    # Pre-process customers with hash-based partitioning for better distribution
    df_customers_hashed = df_customers.withColumn(
        "hash_mod", 
        F.abs(F.hash(F.col("ac_from"))) % total_batches
    )
    
    # Cache the customer list for reuse
    df_customers_hashed.cache()
    df_customers_hashed.count()  # Trigger cache
    
    for batch_num in range(total_batches):
        batch_start_time = datetime.now()
        
        print(f"\n📦 Processing Batch {batch_num + 1}/{total_batches}")
        print(f"   Time: {batch_start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        
        # Get customer batch using hash partitioning
        customer_batch = df_customers_hashed.filter(F.col("hash_mod") == batch_num).select("ac_from")
        
        batch_customer_count = customer_batch.count()
        print(f"   • Batch size: {batch_customer_count:,} customers")
        
        if batch_customer_count == 0:
            print("   ⚠️  No customers in this batch, skipping...")
            continue
        
        # Cache customer batch
        customer_batch.cache()
        customer_batch.count()
        
        # Filter transactions for this batch - use broadcast join for small customer lists
        print("   • Filtering transactions for batch customers...")
        if batch_customer_count < 50000:  # Broadcast small batches
            customer_batch_broadcast = F.broadcast(customer_batch)
            df_batch_transactions = df_transactions.join(
                customer_batch_broadcast, 
                on="ac_from", 
                how="inner"
            )
        else:
            df_batch_transactions = df_transactions.join(
                customer_batch, 
                on="ac_from", 
                how="inner"
            )
        
        # Repartition for better processing
        df_batch_transactions = df_batch_transactions.repartition(50, "ac_from")
        df_batch_transactions.cache()
        
        batch_txn_count = df_batch_transactions.count()
        print(f"   • Transactions in batch: {batch_txn_count:,}")
        
        if batch_txn_count == 0:
            print("   ⚠️  No transactions found for this batch, skipping...")
            customer_batch.unpersist()
            continue
        
        # Create features for this batch
        print("   • Creating features...")
        try:
            feature_engineer = FraudFeatureEngineer(df_batch_transactions)
            
            print("     - Date/time features...")
            feature_engineer.create_date_time_features()
            
            print("     - Amount features...")
            feature_engineer.create_amount_features()
            
            # CHECKPOINT HERE to break the lineage
            print("     - Checkpointing intermediate results...")
            feature_engineer.df = feature_engineer.df.localCheckpoint()
            
            print("     - Channel features...")
            feature_engineer.create_channel_features()
            
            print("     - Balance features...")
            feature_engineer.create_balance_features()
            
            # CHECKPOINT AGAIN before final operations
            print("     - Final checkpoint before save...")
            df_batch_features = feature_engineer.df.localCheckpoint()
            
            # Add batch metadata
            df_batch_features = df_batch_features.withColumn("batch_number", F.lit(batch_num + 1))
            df_batch_features = df_batch_features.withColumn("processing_timestamp", F.current_timestamp())
            
            feature_count = len(df_batch_features.columns)
            print(f"   • Features created: {feature_count} columns")
            
            # Save batch to parquet
            batch_output_path = f"{output_path}/batch_{batch_num + 1:03d}"
            print(f"   • Saving to: {batch_output_path}")
            
            df_batch_features.coalesce(20).write.mode("overwrite").parquet(batch_output_path)
            
            # Clean up cache
            customer_batch.unpersist()
            df_batch_transactions.unpersist()
            df_batch_features.unpersist()
            
            batch_end_time = datetime.now()
            batch_duration = batch_end_time - batch_start_time
            
            print(f"   ✅ Batch {batch_num + 1} completed successfully!")
            print(f"   • Duration: {batch_duration}")
            print(f"   • Customers processed: {batch_customer_count:,}")
            print(f"   • Transactions processed: {batch_txn_count:,}")
            print(f"   • Features created: {feature_count}")
            
        except Exception as e:
            print(f"   ❌ Error processing batch {batch_num + 1}: {str(e)}")
            import traceback
            traceback.print_exc()
            # Clean up on error
            try:
                customer_batch.unpersist()
                df_batch_transactions.unpersist()
            except:
                pass
            continue
    
    # Clean up main cache
    df_customers_hashed.unpersist()
    
    print("\n" + "="*80)
    print("🎉 Batch processing completed!")
    print(f"   • Total batches processed: {total_batches}")
    print(f"   • Output location: {output_path}")
    print("="*80)

# Start with a smaller batch size for testing
print("Starting optimized batch feature engineering...")
process_customer_batches_optimized(
    df_customers=df_july_customer_senders_sample,
    df_transactions=df,
    batch_size=100,  # Start with 500K for faster processing
    output_path="../data/customer_features_batched"
)

Starting optimized batch feature engineering...
🚀 Starting optimized batch processing:
   • Total customers: 1,000,000
   • Batch size: 10,000
   • Total batches: 100
   • Output path: ../data/customer_features_batched



📦 Processing Batch 1/100
   Time: 2025-10-23 10:56:02
   • Batch size: 22,323 customers
   • Filtering transactions for batch customers...


ERROR:root:KeyboardInterrupt while sending command.              (22 + 13) / 35]
Traceback (most recent call last):
  File "/root/miniconda3/envs/fraud/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/root/miniconda3/envs/fraud/lib/python3.10/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/root/miniconda3/envs/fraud/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 